# 📖 Notebook 4: Data Sovereignty Audit

**Goal**: Build an auditing system that proves where data lives and how it moves between regions — essential for GDPR compliance reports.

## Learning Objectives

By the end of this notebook, you'll understand:
- What data sovereignty means and why auditing matters
- How to generate a data residency report across regions
- How to detect compliance violations (data in the wrong region)
- How to build a GDPR compliance dashboard
- Why Microsoft invests heavily in Azure Purview and Compliance Manager

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 08-enterprise/gdpr-paired-regions
docker compose up -d
```

### Visualization

- **Adminer**: http://localhost:8081  
  Compare data across both region databases.

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
from datetime import datetime
from tabulate import tabulate

EU_WEST_CONFIG = {
    "host": "localhost",
    "port": 55433,
    "database": "gdpr_eu_west",
    "user": "demo",
    "password": "demo"
}

EU_NORTH_CONFIG = {
    "host": "localhost",
    "port": 55434,
    "database": "gdpr_eu_north",
    "user": "demo",
    "password": "demo"
}

REGIONS = {
    "eu-west": EU_WEST_CONFIG,
    "eu-north": EU_NORTH_CONFIG
}

def get_connection(region):
    return psycopg2.connect(**REGIONS[region])

# Verify connections
for region in REGIONS:
    conn = get_connection(region)
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM users")
    print(f"✅ {region}: {cur.fetchone()[0]} users")
    conn.close()

## 1. What is Data Sovereignty?

**Data sovereignty** means that data is subject to the laws of the country where it is stored. For GDPR:

- Data stored in the **Netherlands** is subject to Dutch + EU law
- Data stored in **Ireland** is subject to Irish + EU law
- Data stored in the **US** is subject to US law (which may conflict with GDPR!)

### Why Auditing Matters

GDPR Article 30 requires you to maintain a **Record of Processing Activities (ROPA)**. This means you must document:

1. **What** personal data you process
2. **Why** you process it (legal basis)
3. **Where** it is stored (geographic location)
4. **Who** has access to it
5. **How long** you keep it
6. **Where** it is transferred (cross-border flows)

A supervisory authority can ask for this record at any time (Article 30(4)), and failing to produce it is itself an infringement — independent of whether anything was actually done wrong with the data.

Two things worth being precise about before we write code:

- **The ROPA is a document, not a query.** It describes *processing activities* — purposes, categories of data subject, recipients, transfers, retention, security measures. The scripts below produce *evidence* that feeds a ROPA and lets you check reality against it. They do not produce a ROPA.
- **Everything in this notebook is a technical control check.** A control that passes is evidence for one narrow assertion. Passing all of them is not compliance; the bulk of GDPR is about lawful basis, purpose, contracts, and governance, none of which a database can see.

Let's build the tools to generate these reports.

In [ ]:
# ── Audit Tool 1: Data Residency Map ──────────────────────
# Shows where user data is physically stored across regions.

def generate_residency_map():
    """
    Scans all regions and builds a map of where each person's data lives.

    Keyed on email, because that is the only identifier that means the same
    thing in both databases. `users.id` is a per-database SERIAL — using it as
    the map key would merge two different people who happen to share an id.

    We also record the per-region local id and flag any field that DISAGREES
    between regions. Divergence is the interesting signal: it means one region
    got an update the other did not, and an audit that silently keeps
    whichever value it saw first will report a world that does not exist.

    In Azure, discovery at this scale is Microsoft Purview's job. Purview finds
    and classifies data; deciding whether a given placement is lawful is still
    yours.
    """
    residency_map = {}  # email -> record

    for region in REGIONS:
        conn = get_connection(region)
        cur = conn.cursor()
        cur.execute("""
            SELECT id, email, full_name, country_code, home_region
            FROM users
            ORDER BY id
        """)
        for user_id, email, name, country, home in cur.fetchall():
            rec = residency_map.setdefault(email, {
                "local_ids": {}, "name": name, "country": country,
                "home_region": home, "found_in": [], "divergent_fields": set(),
            })
            # Flag disagreement rather than silently keeping the first value.
            for field, value in (("name", name), ("country", country), ("home_region", home)):
                if rec[field] != value:
                    rec["divergent_fields"].add(field)
            rec["local_ids"][region] = user_id
            rec["found_in"].append(region)
        conn.close()

    return residency_map


# Generate and display the map
print("🗺️  DATA RESIDENCY MAP")
print("=" * 80)
print("Shows where each user's PII is physically stored.\n")

rmap = generate_residency_map()

table_data = []
for email, info in sorted(rmap.items()):
    ids = "/".join(str(info["local_ids"].get(r, "—")) for r in REGIONS)
    diverged = ", ".join(sorted(info["divergent_fields"])) or "—"
    table_data.append([
        ids,
        info["name"][:20],
        info["country"],
        info["home_region"],
        ", ".join(info["found_in"]),
        "Yes" if len(info["found_in"]) > 1 else "No",
        diverged,
    ])

print(tabulate(
    table_data,
    headers=["ids (W/N)", "Name", "Country", "Home Region",
             "Data Stored In", "In Both?", "Fields disagreeing"],
    tablefmt="grid"
))

# The map must key on something stable, or it merges strangers.
assert all(len(info["local_ids"]) == len(info["found_in"]) for info in rmap.values())
drifted = {e: sorted(i["local_ids"].items()) for e, i in rmap.items()
           if len(i["local_ids"]) == 2 and len(set(i["local_ids"].values())) > 1}
print(f"\n🔢 People whose local ids differ between regions: {len(drifted)}")
for email, ids in list(drifted.items())[:3]:
    print(f"   {email}: {ids}")
print("   → this is why every cross-region check below joins on email, never on id.")

diverged_people = {e: sorted(i["divergent_fields"]) for e, i in rmap.items() if i["divergent_fields"]}
if diverged_people:
    print(f"\n⚠️  {len(diverged_people)} person(s) have fields that DISAGREE between regions:")
    for email, fields in diverged_people.items():
        print(f"   {email}: {fields}")
    print("   Replication has diverged. Whichever region you queried first would")
    print("   have given you a confident, wrong answer.")
else:
    print("\n✅ No field-level divergence between regions.")

## 2. Compliance Violation Detection

Here is the mistake that makes most homegrown residency audits worthless, and it
is worth understanding before reading the code.

Our `users` table has a `home_region` column. It is tempting to audit it:

```python
allowed = REGION_ALLOWED_COUNTRIES[row.home_region]   # ← the bug
if row.country_code not in allowed:
    violation()
```

That check reads the **label the row carries about itself** and compares it to
the country. It never looks at which database the row was actually found in. So:

| Scenario | Physical location | `home_region` label | Label-based check |
|---|---|---|---|
| Swedish user, correctly placed | eu-north | `eu-north` | ✅ pass (correct) |
| Swedish user, misrouted **and** mislabelled | eu-west | `eu-west` | 🚨 flagged (correct, by luck) |
| Swedish user, misrouted but **correctly labelled** | eu-west | `eu-north` | ✅ pass — **and this is the miss** |
| Swedish user, in a US database, labelled `eu-north` | us-east | `eu-north` | ✅ pass — **catastrophic miss** |

Rows three and four are the realistic failures. A misrouting bug does not
usually corrupt the label too — the label comes from the geo-router's *intent*
and the placement comes from whatever connection the code grabbed. They diverge
precisely when something is wrong, which is when the audit stops working.

An audit whose only input is the data's own self-description is not an audit.
The whole value of asking "where is this row?" is that you connect to each
database and *look*, then compare what you found against what was intended.

So our detector takes three inputs per person:

1. **Where the row physically is** — which database answered when we queried it.
2. **Which region the country maps to** — the geo-router's rule, held outside the data.
3. **What the row claims** — `home_region`, useful only as a fourth thing to cross-check.

And it distinguishes four outcomes:

- **`MISROUTED`** — the row's only home is a region its country does not map to.
- **`CROSS_GEOGRAPHY`** — the row sits in a region outside the legal geography its country requires. This is the one with Chapter V consequences.
- **`MISLABELLED`** — `home_region` disagrees with the geo-router. The data is fine; your audit trail is lying.
- **`REPLICATED`** (informational) — the row is in its home region *and* that region's designated pair. Expected, for DR.


In [ ]:
# ── Audit Tool 2: Compliance Violation Detector ────────────
#
# The three inputs are kept deliberately separate. Two of them are POLICY and
# live in code (or, in a real system, in a versioned config service). Only the
# third comes from the data — which is the point.

# POLICY 1: which region each country's data belongs in.
COUNTRY_TO_REGION = {
    "NL": "eu-west", "BE": "eu-west", "FR": "eu-west",
    "DE": "eu-west", "LU": "eu-west", "AT": "eu-west",
    "IE": "eu-north", "SE": "eu-north", "FI": "eu-north",
    "DK": "eu-north", "NO": "eu-north", "IS": "eu-north",
}

# POLICY 2: which legal geography each region sits in, and which regions are an
# acceptable DR partner. We include a hypothetical 'us-east' so the detector can
# be tested against a genuine cross-border case — the lab only runs two EU
# containers, so without it the CROSS_GEOGRAPHY branch would be dead code that
# nobody ever proves works.
REGION_GEOGRAPHY = {
    "eu-west":  "EU/EEA",
    "eu-north": "EU/EEA",
    "us-east":  "US",        # not deployed here; policy target for testing
}
REGION_PAIR = {"eu-west": "eu-north", "eu-north": "eu-west"}

# Which geography each country's data may occupy at all.
COUNTRY_GEOGRAPHY = {cc: "EU/EEA" for cc in COUNTRY_TO_REGION}

# Backwards-compatible view of the routing policy, for readability.
REGION_ALLOWED_COUNTRIES = {}
for _cc, _r in COUNTRY_TO_REGION.items():
    REGION_ALLOWED_COUNTRIES.setdefault(_r, []).append(_cc)


def detect_violations(extra_rows=()):
    """
    Scans every region for data residency problems.

    `extra_rows` lets a caller inject hypothetical placements as
    (region, email, name, country, home_region) so the cross-geography branch
    can be exercised without deploying a US database. Injected rows are marked
    as such in the output.

    Returns (violations, notices).
    """
    violations, notices = [], []

    # 1. WHERE THINGS ACTUALLY ARE — by querying each database in turn.
    placements = []   # (region, email, name, country, home_region, simulated)
    for region in REGIONS:
        conn = get_connection(region)
        cur = conn.cursor()
        cur.execute("SELECT email, full_name, country_code, home_region FROM users")
        for email, name, country, home in cur.fetchall():
            placements.append((region, email, name, country, home, False))
        conn.close()
    for region, email, name, country, home in extra_rows:
        placements.append((region, email, name, country, home, True))

    # Group by person — email, never by local id (the sequences differ).
    by_person = {}
    for region, email, name, country, home, simulated in placements:
        p = by_person.setdefault(email, {"name": name, "country": country,
                                         "home_labels": set(), "regions": set(),
                                         "simulated": False})
        p["home_labels"].add(home)
        p["regions"].add(region)
        p["simulated"] |= simulated

    for email, p in by_person.items():
        country = p["country"]
        expected_region = COUNTRY_TO_REGION.get(country)
        required_geo = COUNTRY_GEOGRAPHY.get(country)
        tag = " [simulated]" if p["simulated"] else ""

        if expected_region is None:
            violations.append({
                "type": "UNMAPPED_COUNTRY", "severity": "HIGH",
                "email": email, "name": p["name"], "country": country,
                "found_in": sorted(p["regions"]),
                "reason": f"Country {country} has no region assignment — nobody decided "
                          f"where this person's data is allowed to live{tag}",
            })
            continue

        allowed_regions = {expected_region, REGION_PAIR.get(expected_region)} - {None}

        # ── CHECK A: physical location vs required legal geography ──
        for region in sorted(p["regions"]):
            geo = REGION_GEOGRAPHY.get(region, "UNKNOWN")
            if required_geo and geo != required_geo:
                violations.append({
                    "type": "CROSS_GEOGRAPHY", "severity": "CRITICAL",
                    "email": email, "name": p["name"], "country": country,
                    "found_in": [region],
                    "reason": f"Row is physically in {region} ({geo}) but {country} data "
                              f"requires {required_geo}. This is a Chapter V transfer and "
                              f"needs a lawful transfer mechanism and a record{tag}",
                })

        # ── CHECK B: physical location vs the routing policy ──
        # Two distinct failures, both of which the label-based check misses:
        #   B1 no copy in the region the country actually routes to
        #   B2 a copy somewhere outside {home region, its pair}
        stray = p["regions"] - allowed_regions
        if expected_region not in p["regions"]:
            violations.append({
                "type": "MISROUTED", "severity": "HIGH",
                "email": email, "name": p["name"], "country": country,
                "found_in": sorted(p["regions"]),
                "reason": f"Country {country} routes to {expected_region}, but no copy "
                          f"exists there. The data lives only in {sorted(p['regions'])}"
                          f"{tag}",
            })
        if stray and not any(v["type"] == "CROSS_GEOGRAPHY" and v["email"] == email
                             for v in violations):
            violations.append({
                "type": "MISROUTED", "severity": "HIGH",
                "email": email, "name": p["name"], "country": country,
                "found_in": sorted(stray),
                "reason": f"Country {country} routes to {expected_region} "
                          f"(pair: {REGION_PAIR.get(expected_region)}), but a copy was "
                          f"found in {sorted(stray)}{tag}",
            })

        # ── CHECK C: the label vs the policy ──
        # Deliberately separate from A and B: a wrong label is an evidence
        # problem, not a placement problem, and conflating them is how the
        # original detector ended up auditing nothing but the label.
        bad_labels = {lbl for lbl in p["home_labels"] if lbl != expected_region}
        if bad_labels:
            violations.append({
                "type": "MISLABELLED", "severity": "MEDIUM",
                "email": email, "name": p["name"], "country": country,
                "found_in": sorted(p["regions"]),
                "reason": f"home_region label(s) {sorted(bad_labels)} disagree with the "
                          f"geo-router, which says {country} → {expected_region}. The data "
                          f"may be fine; the audit trail is not{tag}",
            })

        # ── Informational: expected DR replication ──
        if p["regions"] == allowed_regions:
            notices.append({
                "email": email, "name": p["name"], "severity": "INFO",
                "reason": f"Present in {expected_region} and its pair "
                          f"{REGION_PAIR.get(expected_region)} — expected for DR, "
                          f"same geography, no transfer",
            })

    return violations, notices


def print_scan(violations, notices, show_notices=3):
    if violations:
        print(f"\n🚨 {len(violations)} VIOLATION(S) FOUND:")
        for v in sorted(violations, key=lambda x: x["severity"]):
            print(f"   ❌ [{v['severity']}/{v['type']}] {v['name']} ({v['email']})")
            print(f"      {v['reason']}")
    else:
        print("\n✅ No residency violations found in the live databases.")
    if notices:
        print(f"\nℹ️  {len(notices)} informational notice(s); showing {show_notices}:")
        for n in notices[:show_notices]:
            print(f"   📋 {n['name']}: {n['reason']}")


print("🔍 COMPLIANCE VIOLATION SCAN (baseline)")
print("=" * 60)
violations, notices = detect_violations()
print_scan(violations, notices)

print("\n💡 If you ran Notebook 3 against these same containers, expect findings")
print("   here — and they are correct findings, not noise. Pseudonymising a user")
print("   in one region only leaves the replica holding the original identified")
print("   record, and the audit is supposed to notice that. On fresh containers")
print("   this baseline is clean.")


In [ ]:
# ── Testing the detector against the cases that matter ─────────────
# A detector nobody has tried to fool is a detector that has never been tested.
# We plant three specific violations, including the one the label-based check
# silently passed, and require the detector to catch each of them.

print("🧪 DETECTOR TEST SUITE")
print("=" * 62)

TEST_EMAILS = ["viol.mislabelled@example.se", "viol.correctlabel@example.se"]

def cleanup_test_rows():
    for region in REGIONS:
        conn = get_connection(region)
        cur = conn.cursor()
        cur.execute("DELETE FROM users WHERE email = ANY(%s)", (TEST_EMAILS,))
        conn.commit()
        conn.close()

cleanup_test_rows()

conn = get_connection("eu-west")
cur = conn.cursor()
# Case 1 — Swedish user in eu-west, ALSO mislabelled as eu-west.
#          The old label-based check caught this one, which is why it looked
#          like it worked.
cur.execute("""
    INSERT INTO users (email, full_name, country_code, home_region, consent_given)
    VALUES ('viol.mislabelled@example.se', 'Mislabelled Maja', 'SE', 'eu-west', TRUE)
""")
# Case 2 — Swedish user in eu-west, but CORRECTLY labelled home_region='eu-north'.
#          The row is in the wrong database and says so honestly. The old check
#          compared country against the label, found SE→eu-north consistent, and
#          reported no violation. This is the realistic misrouting bug.
cur.execute("""
    INSERT INTO users (email, full_name, country_code, home_region, consent_given)
    VALUES ('viol.correctlabel@example.se', 'Correct-Label Klara', 'SE', 'eu-north', TRUE)
""")
conn.commit()
conn.close()

# Case 3 — a Dutch user's row sitting in a US region. We have no US container,
#          so we inject it as a hypothetical placement. This is the only case
#          with Chapter V consequences, and it must be the loudest.
simulated_us = [("us-east", "viol.transfer@example.nl", "Transferred Tom", "NL", "eu-west")]

violations, notices = detect_violations(extra_rows=simulated_us)
print_scan(violations, notices, show_notices=2)

# ── Now require each planted violation to have been caught ──
found = {}
for v in violations:
    found.setdefault(v["email"], set()).add(v["type"])

print("\n" + "=" * 62)
print("📋 TEST RESULTS")
print("=" * 62)

assert "MISROUTED" in found.get("viol.mislabelled@example.se", set()), (
    "case 1 (misrouted AND mislabelled) was not flagged as MISROUTED"
)
assert "MISLABELLED" in found.get("viol.mislabelled@example.se", set()), (
    "case 1 should also be flagged as MISLABELLED"
)
print("   ✅ case 1  misrouted + mislabelled  → MISROUTED + MISLABELLED")

assert "MISROUTED" in found.get("viol.correctlabel@example.se", set()), (
    "case 2 was NOT caught. This is the exact miss the label-based detector had: "
    "a Swedish row physically in eu-west whose home_region correctly says "
    "eu-north. Comparing country to the label finds them consistent and reports "
    "a clean bill of health while the data sits in the wrong database."
)
assert "MISLABELLED" not in found.get("viol.correctlabel@example.se", set()), (
    "case 2's label is correct — flagging it as mislabelled would confuse a "
    "placement problem with an evidence problem"
)
print("   ✅ case 2  misrouted, label CORRECT → MISROUTED  ← the old detector missed this")

assert "CROSS_GEOGRAPHY" in found.get("viol.transfer@example.nl", set()), (
    "case 3 (NL data in a US region) was not flagged as CROSS_GEOGRAPHY"
)
assert any(v["severity"] == "CRITICAL" for v in violations
           if v["email"] == "viol.transfer@example.nl"), \
    "a cross-border transfer must be the highest severity the detector emits"
print("   ✅ case 3  NL row in us-east        → CROSS_GEOGRAPHY / CRITICAL")

print("\n🔑 The lesson in one line:")
print("   Audit where the data IS, not what the data says about itself.")
print("   The label is a fourth thing to cross-check, never the source of truth.")

cleanup_test_rows()
print("\n🧹 Test rows removed.")


## 2b. The audit that matters most: did the erasures actually finish?

Notebook 3 ended with an uncomfortable demonstration: an erasure ran against the
primary, the `erasure_requests` row was set to `completed`, and the paired
replica still held the person's name, phone and address.

That state is invisible to every check we have written so far. The residency
detector is happy — the surviving row is in an EU region its country maps to.
The erasure-request report is happy — the status column says `completed`.

**Nothing cross-checks the claim against the data.**

This is the general shape of the most dangerous audit failure: a control that
reads a status field written by the very process whose success it is supposed to
verify. Statuses are self-reported. An audit that trusts them audits nothing.

So the check has to be: for every erasure request marked complete, in every
region, go and look for the person.


In [ ]:
# ── Audit Tool 2b: Erasure completeness ────────────────────────────
# Cross-check every 'completed' erasure request against the actual data.

def audit_erasure_completeness():
    """
    For every erasure request marked completed in ANY region, verify that no
    trace of the data subject survives in ANY region.

    Note the cross product: a request completed in eu-west says nothing about
    eu-north, and the request rows themselves do not replicate. So we collect
    the union of all completed requests, then sweep every region for each one.
    """
    # 1. Every subject anyone claims to have erased.
    # email -> {region: set of statuses}. A set, not the last row seen: the same
    # subject can have several request rows in one region (a retried request, or
    # a second one raised after a restore resurrected them), and quietly keeping
    # whichever the query returned last is how audits end up reporting fiction.
    claimed = {}
    for region in REGIONS:
        conn = get_connection(region)
        cur = conn.cursor()
        cur.execute("""
            SELECT id, user_email, status, completed_at
            FROM erasure_requests
        """)
        for req_id, email, status, completed_at in cur.fetchall():
            claimed.setdefault(email, {}).setdefault(region, set()).add(status)
        conn.close()

    # 2. For each of them, look for surviving data in every region.
    findings = []
    for email, per_region in claimed.items():
        completed_in = [r for r, statuses in per_region.items() if "completed" in statuses]
        open_in = [r for r, statuses in per_region.items()
                   if statuses - {"completed", "denied"}]
        if not completed_in:
            continue

        residue = {}
        for region in REGIONS:
            conn = get_connection(region)
            cur = conn.cursor()
            cur.execute("SELECT id FROM users WHERE email = %s", (email,))
            row = cur.fetchone()
            tables = []
            if row:
                local_id = row[0]
                tables.append("users")
                for table in ("addresses", "orders", "consent_log"):
                    cur.execute(f"SELECT COUNT(*) FROM {table} WHERE user_id = %s", (local_id,))
                    if cur.fetchone()[0]:
                        tables.append(table)
            conn.close()
            if tables:
                residue[region] = tables

        # No request row in a region at all is its own gap: nobody even tried.
        never_requested = [r for r in REGIONS if r not in per_region]

        findings.append({
            "email": email,
            "completed_in": sorted(completed_in),
            "still_open_in": sorted(open_in),
            "never_requested_in": never_requested,
            "residue": residue,
            "complete": not residue and not never_requested and not open_in,
        })
    return findings


print("🔍 ERASURE COMPLETENESS AUDIT")
print("=" * 62)

findings = audit_erasure_completeness()
if not findings:
    print("\n   ℹ️  No completed erasure requests found.")
    print("      (Run Notebook 3 against these containers to generate some.)")
else:
    incomplete = [f for f in findings if not f["complete"]]
    for f in findings:
        if f["complete"]:
            print(f"\n   ✅ {f['email']}")
            print(f"      completed in {f['completed_in']}, no residue in any region")
        else:
            print(f"\n   🚨 {f['email']} — REPORTED COMPLETE, NOT COMPLETE")
            print(f"      status 'completed' in : {f['completed_in']}")
            if f["residue"]:
                for region, tables in f["residue"].items():
                    print(f"      ❌ data still present in {region}: {tables}")
            if f["still_open_in"]:
                print(f"      ❌ a request is still un-closed in: {f['still_open_in']}")
            if f["never_requested_in"]:
                print(f"      ❌ no erasure request was ever raised in: {f['never_requested_in']}")

    print("\n" + "-" * 62)
    print(f"   {len(findings) - len(incomplete)}/{len(findings)} erasures verifiably complete")
    if incomplete:
        print(f"   {len(incomplete)} request(s) are recorded as fulfilled but are not.")
        print( "   Each one is an ongoing Article 17 failure AND a false statement")
        print( "   made to the data subject when you confirmed the erasure.")

# ── Prove the audit can actually detect the failure ───────────────
# Plant the exact state Notebook 3 demonstrated — erased in the primary,
# surviving in the replica, request marked completed — and require a catch.
PROBE = "audit.probe@example.nl"
conn = get_connection("eu-north")
cur = conn.cursor()
cur.execute("DELETE FROM users WHERE email = %s", (PROBE,))
cur.execute("""
    INSERT INTO users (email, full_name, phone, country_code, home_region, consent_given)
    VALUES (%s, 'Probe Pieter', '+31-6-9999-0000', 'NL', 'eu-west', TRUE)
""", (PROBE,))
conn.commit(); conn.close()

conn = get_connection("eu-west")
cur = conn.cursor()
cur.execute("DELETE FROM erasure_requests WHERE user_email = %s", (PROBE,))
cur.execute("""
    INSERT INTO erasure_requests (user_id, user_email, reason, status, completed_at, completed_by)
    VALUES (-1, %s, 'audit self-test', 'completed', NOW(), 'system')
""", (PROBE,))
conn.commit(); conn.close()

probe_findings = {f["email"]: f for f in audit_erasure_completeness()}
assert PROBE in probe_findings, "the audit did not consider the planted request at all"
assert not probe_findings[PROBE]["complete"], (
    "the audit reported a verified-complete erasure for a data subject whose "
    "record is demonstrably still sitting in the paired region. This check is "
    "the only thing standing between 'status = completed' and the truth."
)
assert "eu-north" in probe_findings[PROBE]["residue"], \
    f"residue should have been located in eu-north; got {probe_findings[PROBE]['residue']}"
print(f"\n🧪 Self-test: planted an incomplete erasure — audit caught it. ✅")

# Clean up the probe.
for region in REGIONS:
    conn = get_connection(region)
    cur = conn.cursor()
    cur.execute("DELETE FROM users WHERE email = %s", (PROBE,))
    cur.execute("DELETE FROM erasure_requests WHERE user_email = %s", (PROBE,))
    conn.commit(); conn.close()
print("🧹 Probe removed.")


## 3. Consent Audit Report

Careful here, because this is the second-most-common GDPR misconception in
engineering (after "data must stay in the EU").

**Consent is one of six lawful bases, not the default one.** Article 6(1) lists
consent, contract, legal obligation, vital interests, public task, and legitimate
interests. For most of what a shop does with a customer's data — taking the
order, shipping it, keeping the invoice — the basis is **contract** or **legal
obligation**, and asking for consent would actually be *wrong*: consent must be
freely given and freely withdrawable, and you cannot let someone withdraw
consent to being sent the thing they bought.

Consequences for the audit we are about to write:

- A user with `consent_given = FALSE` is **not automatically a compliance risk**.
  They may be perfectly lawfully served under a different basis.
- "Consent rate" is therefore **not a compliance metric**. A 100% consent rate is
  as likely to indicate a coerced cookie wall as a healthy one.
- What *is* auditable: for each **purpose**, is there a recorded basis? Where the
  basis is consent, can you show it was given, when, for what, and that it can be
  withdrawn as easily as it was given (Article 7(3))?

Our schema only has a boolean and a purpose-tagged log, so it can answer the
narrow question — *for purposes that rely on consent, do we hold a record?* —
and nothing more. The report below is written to claim only that.


In [ ]:
# ── Audit Tool 3: Consent Status Report ───────────────────

# Which purposes in our system rely on CONSENT as their lawful basis, and which
# rely on something else. Without this mapping the report cannot say anything
# meaningful — it can only count a boolean.
PURPOSE_LAWFUL_BASIS = {
    "essential":  "contract",            # delivering the service they bought
    "marketing":  "consent",             # needs consent (and ePrivacy rules too)
    "analytics":  "consent",             # non-essential tracking
    # statutory invoice retention would be "legal_obligation"
}
CONSENT_PURPOSES = {p for p, b in PURPOSE_LAWFUL_BASIS.items() if b == "consent"}


def generate_consent_report():
    """
    Reports on consent RECORDS across all regions.

    Article 7(1) requires the controller to be able to demonstrate that consent
    was given, where consent is the basis relied on. This report answers that
    narrow question. It deliberately does NOT compute a "consent rate" as a
    health metric: users without consent may be lawfully processed under
    contract, legal obligation or legitimate interests, and a high consent rate
    can just as easily indicate a coerced consent flow.
    """
    report = {"total_users": 0, "consented": 0, "no_consent": 0, "by_region": {}}

    for region in REGIONS:
        conn = get_connection(region)
        cur = conn.cursor()

        # Overall consent stats
        cur.execute("""
            SELECT
                COUNT(*) as total,
                COUNT(*) FILTER (WHERE consent_given = TRUE) as consented,
                COUNT(*) FILTER (WHERE consent_given = FALSE OR consent_given IS NULL) as no_consent
            FROM users
            WHERE home_region = %s
        """, (region,))
        stats = cur.fetchone()

        report["by_region"][region] = {
            "total": stats[0],
            "consented": stats[1],
            "no_consent": stats[2]
        }
        report["total_users"] += stats[0]
        report["consented"] += stats[1]
        report["no_consent"] += stats[2]

        # Users with no consent flag. NOT presented as a risk — see the markdown
        # above. Listed so you can check each one has *some* lawful basis.
        cur.execute("""
            SELECT id, email, full_name, country_code
            FROM users
            WHERE home_region = %s AND (consent_given = FALSE OR consent_given IS NULL)
        """, (region,))
        report["by_region"][region]["no_consent_users"] = cur.fetchall()

        # Consent by purpose
        cur.execute("""
            SELECT purpose, action, COUNT(*) as cnt
            FROM consent_log cl
            JOIN users u ON cl.user_id = u.id
            WHERE u.home_region = %s
            GROUP BY purpose, action
            ORDER BY purpose, action
        """, (region,))
        report["by_region"][region]["consent_details"] = cur.fetchall()

        conn.close()

    return report


# Generate and display
print("📋 GDPR CONSENT AUDIT REPORT")
print(f"   Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

report = generate_consent_report()

# Summary
consent_rate = (report['consented'] / report['total_users'] * 100) if report['total_users'] > 0 else 0
print(f"\n📊 Overall Summary:")
print(f"   Total users: {report['total_users']}")
print(f"   With a consent flag set: {report['consented']} ({consent_rate:.0f}%)")
print(f"   Without: {report['no_consent']}")
print( "   ⚠️  This percentage is descriptive, not a score. Users without consent")
print( "       are not automatically a problem — check they have another basis.")
print(f"   Purposes relying on consent in this system: {sorted(CONSENT_PURPOSES)}")

# Per-region details
for region, data in report["by_region"].items():
    print(f"\n📦 {region.upper()}:")
    print(f"   Users: {data['total']} | Consented: {data['consented']} | No consent: {data['no_consent']}")

    if data["no_consent_users"]:
        print(f"   📌 Users with no consent flag (verify each has a lawful basis):")
        for u in data["no_consent_users"]:
            print(f"      - {u[2]} ({u[1]}) — country: {u[3]}")

    if data["consent_details"]:
        print(f"   Consent breakdown:")
        for purpose, action, count in data["consent_details"]:
            icon = "✅" if action == "granted" else "🚫"
            basis = PURPOSE_LAWFUL_BASIS.get(purpose, "UNMAPPED")
            flag = "" if basis != "UNMAPPED" else "  ⚠️ no lawful basis recorded for this purpose!"
            print(f"      {icon} {purpose}: {action} ({count})  [basis: {basis}]{flag}")

# Every purpose we actually log must have a declared lawful basis. A purpose
# appearing in the data but not in the policy map means someone started
# processing for a reason nobody wrote down — which is an Article 30 gap and
# usually an Article 6 one too.
logged_purposes = set()
for region in REGIONS:
    conn = get_connection(region)
    cur = conn.cursor()
    cur.execute("SELECT DISTINCT purpose FROM consent_log")
    logged_purposes.update(r[0] for r in cur.fetchall())
    conn.close()
unmapped = logged_purposes - set(PURPOSE_LAWFUL_BASIS)
assert not unmapped, (
    f"purposes {sorted(unmapped)} appear in consent_log but have no declared "
    f"lawful basis. You are processing for a purpose nobody has justified."
)
print(f"\n✅ All logged purposes {sorted(logged_purposes)} have a declared lawful basis.")

## 4. Erasure Request Tracking

If your organization has received erasure requests, you need to track them and show they were handled in time.

The deadline is **Article 12(3): without undue delay and in any event within one month of receipt** — extendable by two further months for complex or numerous requests, provided you inform the data subject within the first month and explain why. "30 days" is a reasonable internal SLA and is what we track below, but note it is stricter than the law in most months and does not model the extension.

Note also what this report can and cannot tell you. It reads `status` and `completed_at` — fields written by the erasure process about itself. Section 2b is the report that actually checks whether the erasure happened.

In [ ]:
# ── Audit Tool 4: Erasure Request Status Report ───────────

def generate_erasure_report():
    """Generates a report of all erasure requests across regions."""
    all_requests = []

    for region in REGIONS:
        conn = get_connection(region)
        cur = conn.cursor()
        cur.execute("""
            SELECT id, user_id, user_email, reason, status,
                   requested_at, completed_at, completed_by
            FROM erasure_requests
            ORDER BY requested_at DESC
        """)

        for row in cur.fetchall():
            days_elapsed = None
            if row[5]:  # requested_at
                if row[6]:  # completed_at
                    days_elapsed = (row[6] - row[5]).days
                else:
                    days_elapsed = (datetime.now() - row[5]).days

            all_requests.append({
                "region": region,
                "id": row[0],
                "user_email": row[2],
                "reason": row[3],
                "status": row[4],
                "requested_at": row[5],
                "completed_at": row[6],
                "days_elapsed": days_elapsed,
                # Internal SLA of 30 days, stricter than the statutory "one
                # month". Does NOT model the Article 12(3) two-month extension.
                "within_sla": days_elapsed is not None and days_elapsed <= 30
            })
        conn.close()

    return all_requests


print("📋 ERASURE REQUEST STATUS REPORT")
print(f"   Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 70)

requests = generate_erasure_report()

if requests:
    table_data = []
    for r in requests:
        status_icon = {
            "completed": "✅",
            "processing": "🔄",
            "pending": "⏳",
            "denied": "❌"
        }.get(r["status"], "❓")

        deadline_status = ""
        if r["days_elapsed"] is not None:
            if r["status"] == "completed":
                deadline_status = f"{r['days_elapsed']}d (OK)" if r["within_sla"] else f"{r['days_elapsed']}d (OVERDUE!)"
            else:
                remaining = 30 - r["days_elapsed"]
                deadline_status = f"{remaining}d left" if remaining > 0 else "OVERDUE!"

        table_data.append([
            r["region"],
            f"#{r['id']}",
            r["user_email"],
            f"{status_icon} {r['status']}",
            deadline_status
        ])

    print(tabulate(
        table_data,
        headers=["Region", "Request", "User Email", "Status", "30-Day SLA"],
        tablefmt="grid"
    ))
    print("\n⚠️  'completed' here is self-reported by the erasure process.")
    print("    Section 2b is what verifies it against the actual data.")
else:
    print("\nNo erasure requests found.")
    print("(Run Notebook 3 first to generate erasure requests)")

## 5. Data Movement Audit Trail

Every time data moves between regions, it should be logged. This is critical for proving compliance during a DPA audit:

In [ ]:
# ── Audit Tool 5: Data Movement Log ───────────────────────

def generate_movement_report():
    """Shows all recorded data movements between regions."""
    movements = []

    for region in REGIONS:
        conn = get_connection(region)
        cur = conn.cursor()
        cur.execute("""
            SELECT drl.user_id, drl.action, drl.source_region,
                   drl.target_region, drl.table_name, drl.reason,
                   drl.created_at
            FROM data_residency_log drl
            ORDER BY drl.created_at DESC
            LIMIT 20
        """)

        for row in cur.fetchall():
            movements.append({
                "logged_in": region,
                "user_id": row[0],
                "action": row[1],
                "source": row[2],
                "target": row[3] or "N/A",
                "table": row[4],
                "reason": row[5],
                "timestamp": row[6]
            })
        conn.close()

    # Sort by timestamp
    movements.sort(key=lambda x: x["timestamp"] if x["timestamp"] else datetime.min, reverse=True)
    return movements


print("📋 DATA MOVEMENT AUDIT TRAIL")
print(f"   Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)

movements = generate_movement_report()

if movements:
    action_icons = {
        "write": "✏️",
        "replicate": "🔄",
        "delete": "🗑️",
        "access": "👁️"
    }

    table_data = []
    for m in movements[:15]:  # Show most recent 15
        icon = action_icons.get(m["action"], "❓")
        table_data.append([
            str(m["timestamp"])[:19] if m["timestamp"] else "N/A",
            f"{icon} {m['action']}",
            f"User {m['user_id']}",
            m["source"],
            m["target"],
            m["table"],
        ])

    print(tabulate(
        table_data,
        headers=["Timestamp", "Action", "User", "Source", "Target", "Table"],
        tablefmt="grid"
    ))

    print(f"\n📊 Summary:")
    action_counts = {}
    for m in movements:
        action_counts[m["action"]] = action_counts.get(m["action"], 0) + 1
    for action, count in sorted(action_counts.items()):
        icon = action_icons.get(action, "❓")
        print(f"   {icon} {action}: {count} event(s)")
else:
    print("\nNo data movements recorded yet.")
    print("(Run Notebooks 1-3 to generate movement data)")

## 6. Technical Controls Report

Time to combine the checks. And time to be very careful about what we call the
result, because this is where teaching labs habitually go wrong.

### Why this is not a "compliance score"

An earlier version of this notebook printed:

```
📊 OVERALL COMPLIANCE SCORE: 92/100 (Grade: A)
🎉 Excellent! Your system is well-prepared for a GDPR audit.
```

Every part of that is a problem:

- **You cannot average controls into a compliance number.** The arithmetic mean
  of "residency ✅, consent ⚠️, erasure ✅, audit trail ✅" was 92. But a single
  unlawful international transfer, or one erasure that never happened, is an
  infringement on its own. Controls are **conjunctive** — you need all of them —
  and averaging lets three passes hide one failure. Note that in the original
  run the residency check reported ✅ *while a data subject who had requested
  erasure was still sitting in the replica.*
- **A grade implies an authority.** Nothing in a notebook can grade GDPR
  compliance. The bodies that can are supervisory authorities, and the closest
  thing to a formal grade is an Article 42 certification from an accredited body
  against an approved scheme — which is not what this is.
- **"Well-prepared for a GDPR audit" is a claim about your whole organisation.**
  These four scripts see one schema in two databases. They cannot see your lawful
  bases, your ROPA, your DPIAs, your processor contracts, your transfer impact
  assessments, your retention policy, your security measures, your breach
  procedures, your DPO, or your training.

### What we print instead

A **technical controls report**: each check is named, its scope is stated, and it
is PASS / FAIL / NOT ASSESSED. No averaging, no grade. Any FAIL makes the overall
result FAIL, because that is how conjunctive requirements work. And the report
ends with an explicit, un-skippable list of the things it did **not** look at —
because the most dangerous output of a compliance tool is a green tick with
unstated scope.


In [ ]:
# ── Technical Controls Report ──────────────────────────────────────
# Named checks, explicit scope, conjunctive result. No score, no grade.

def generate_controls_report():
    """
    Runs each technical control and reports PASS / FAIL / NOT ASSESSED.

    Returns (results, overall_pass). Deliberately does NOT return a number:
    the moment a compliance check emits a score, somebody puts it on a
    dashboard and starts optimising the average.
    """
    print("╔" + "═" * 66 + "╗")
    print("║" + "  TECHNICAL CONTROLS REPORT".center(66) + "║")
    print("║" + "  (not a compliance assessment — see scope note at end)".center(66) + "║")
    print("║" + f"  Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}".center(66) + "║")
    print("╚" + "═" * 66 + "╝")

    results = []

    def record(name, scope, status, detail):
        results.append({"name": name, "scope": scope, "status": status, "detail": detail})
        icon = {"PASS": "✅", "FAIL": "❌", "NOT ASSESSED": "➖"}[status]
        print(f"\n{icon} {name}: {status}")
        print(f"   scope : {scope}")
        print(f"   result: {detail}")

    # ── C1: physical data residency ──
    violations, _ = detect_violations()
    critical = [v for v in violations if v["severity"] == "CRITICAL"]
    record(
        "C1 Data residency (physical placement)",
        "users table only, in the 2 regions this notebook can reach",
        "FAIL" if violations else "PASS",
        (f"{len(violations)} violation(s), {len(critical)} cross-geography"
         if violations else "every user row is in a region its country maps to"),
    )

    # ── C2: erasure actually completed ──
    findings = audit_erasure_completeness()
    incomplete = [f for f in findings if not f["complete"]]
    if not findings:
        record("C2 Erasure completeness", "completed erasure_requests vs live data",
               "NOT ASSESSED", "no completed erasure requests exist to verify")
    else:
        record(
            "C2 Erasure completeness",
            "completed erasure_requests vs live data in both regions; "
            "backups/caches/third parties NOT checked",
            "FAIL" if incomplete else "PASS",
            (f"{len(incomplete)} of {len(findings)} requests report complete but data survives"
             if incomplete else f"all {len(findings)} verified against the data"),
        )

    # ── C3: lawful basis declared for every logged purpose ──
    logged = set()
    for region in REGIONS:
        conn = get_connection(region)
        cur = conn.cursor()
        cur.execute("SELECT DISTINCT purpose FROM consent_log")
        logged.update(r[0] for r in cur.fetchall())
        conn.close()
    missing_basis = sorted(logged - set(PURPOSE_LAWFUL_BASIS))
    record(
        "C3 Lawful basis declared per purpose",
        "purposes appearing in consent_log; does NOT assess whether the "
        "declared basis is CORRECT, or whether consent was validly obtained",
        "FAIL" if missing_basis else "PASS",
        f"undeclared purposes: {missing_basis}" if missing_basis
        else f"all {len(logged)} logged purposes have a declared basis",
    )

    # ── C4: erasure SLA ──
    erasure_requests = generate_erasure_report()
    overdue = [r for r in erasure_requests
               if r["status"] not in ("completed", "denied")
               and r["days_elapsed"] and r["days_elapsed"] > 30]
    if not erasure_requests:
        record("C4 Erasure response SLA", "erasure_requests timestamps",
               "NOT ASSESSED", "no erasure requests on record")
    else:
        record(
            "C4 Erasure response SLA (30-day internal target)",
            "open requests only; does not model the Art. 12(3) extension",
            "FAIL" if overdue else "PASS",
            f"{len(overdue)} overdue" if overdue
            else f"all {len(erasure_requests)} request(s) within the internal target",
        )

    # ── C5: audit trail present ──
    movements = generate_movement_report()
    has_write = any(m["action"] == "write" for m in movements)
    has_delete = any(m["action"] == "delete" for m in movements)
    if not movements:
        record(
            "C5 Data movement audit trail",
            "data_residency_log in both regions",
            "NOT ASSESSED",
            "no movement events recorded yet — run Notebooks 1-3 against these "
            "containers first",
        )
    else:
        record(
            "C5 Data movement audit trail",
            "data_residency_log only; READ access is not logged anywhere, which is "
            "a significant gap for Art. 30 and Art. 32",
            "PASS" if (has_write and has_delete) else "FAIL",
            f"{len(movements)} event(s); writes logged={has_write}, deletes logged={has_delete}",
        )

    # ── Overall: conjunctive, never averaged ──
    failed = [r for r in results if r["status"] == "FAIL"]
    unassessed = [r for r in results if r["status"] == "NOT ASSESSED"]
    overall_pass = not failed

    print("\n" + "=" * 68)
    if overall_pass:
        print("RESULT: all assessed technical controls PASS"
              + (f" ({len(unassessed)} not assessed)" if unassessed else ""))
    else:
        print(f"RESULT: FAIL — {len(failed)} control(s) failing: "
              + ", ".join(r["name"].split()[0] for r in failed))
        print("        One failing control is a failure. Controls are conjunctive;")
        print("        there is no average that makes a missed erasure acceptable.")
    print("=" * 68)

    print("""
📌 SCOPE OF THIS REPORT — read before quoting it anywhere

This checks five technical controls against two Postgres databases. Passing
them means those five things are behaving. It does NOT mean the organisation
complies with GDPR, and this report must not be presented as evidence that it
does.

Not assessed here, and mostly not assessable by any script:
  • lawful basis in fact (Art. 6) — whether the declared basis is the right one
  • validity of consent (Art. 7) — freely given, specific, informed, unambiguous
  • Record of Processing Activities (Art. 30) — a document, not a query
  • DPIAs (Art. 35), and prior consultation where required (Art. 36)
  • processor contracts (Art. 28) and sub-processor chains
  • transfer mechanisms and transfer impact assessments (Chapter V)
  • security of processing (Art. 32) — this lab has no encryption, no TLS, no
    access control, one shared superuser, and logs no reads
  • breach detection and the Art. 33/34 notification process
  • retention schedules and their enforcement
  • data minimisation and purpose limitation (Art. 5)
  • data subject rights other than access and erasure
  • backups, caches, search indexes, analytics copies and third-party systems

The correct sentence to write in a design doc is:
  "These technical controls pass, which supports our compliance position."
Never:
  "The system is GDPR compliant."
""")
    return results, overall_pass


results, overall_pass = generate_controls_report()

# Whatever the report says, it must be internally consistent — a green overall
# result while a named control is failing is exactly the bug this section is
# about.
assert overall_pass == all(r["status"] != "FAIL" for r in results), \
    "the overall result disagrees with the individual controls"
assert not any(r["status"] == "PASS" and "compliance" in r["detail"].lower()
               for r in results), \
    "no individual control may describe itself as establishing compliance"


## 7. The 72-Hour Breach Notification (Article 33)

Article 33(1): a personal-data breach must be notified to the supervisory
authority **without undue delay and, where feasible, not later than 72 hours
after having become aware of it** — unless the breach is *unlikely to result in
a risk to the rights and freedoms of natural persons*. If you miss 72 hours you
must notify anyway, with reasons for the delay. Article 34 separately requires
notifying the **data subjects** where the risk to them is *high*.

> ⚠️ **A correction to what this notebook used to say.** An earlier version
> claimed British Airways' £20m ICO fine in 2020 was "partly because they took
> too long to notify". That is backwards. The penalty was for **failing to
> process personal data securely** — the ICO found BA had inadequate security
> measures that would have prevented the 2018 attack. BA's *prompt* notification
> to the ICO and to affected customers was cited as a **mitigating** factor that
> helped reduce the penalty from the £183m originally proposed. The real
> criticism on timing was that BA did not **detect** the attack for over two
> months, and learned of it from a third party.
>
> The lesson is actually more useful this way round: the 72-hour clock starts
> when you *become aware*, so an organisation with poor detection does not get a
> generous deadline — it gets a deadline that starts late, after the damage is
> done, with an obligation to explain why.

What counts as a breach?
- Unauthorized **access** to PII (SQL injection, stolen credentials)
- **Loss** of PII (lost laptop, deleted backups)
- **Disclosure** of PII to the wrong people (email sent to wrong list)

Below is a tiny helper that starts the 72-hour clock the moment a breach is
detected, so you always know how much of the reporting window is left.


In [ ]:
# ── Audit Tool 6: 72-hour breach-notification clock ────────
from datetime import timedelta

def check_breach_sla(breach_detected_at, now=None):
    """
    Given the moment a breach was detected, return how much of the 72-hour
    Article 33 reporting window is left.

    The clock starts at AWARENESS, not at compromise. `now` is injectable so
    this is testable without depending on the wall clock.
    """
    now = now or datetime.now()
    deadline = breach_detected_at + timedelta(hours=72)
    remaining = deadline - now
    hours_left = remaining.total_seconds() / 3600
    return {
        "detected_at": breach_detected_at,
        "deadline": deadline,
        "hours_remaining": round(hours_left, 1),
        "status": "ON TRACK" if hours_left > 12 else "URGENT" if hours_left > 0 else "OVERDUE",
    }

# Simulate a breach detected 10 hours ago
simulated_breach = datetime.now() - timedelta(hours=10)
sla = check_breach_sla(simulated_breach)

print("🚨 SIMULATED BREACH — Article 33 countdown")
print("=" * 50)
print(f"   Detected at     : {sla['detected_at']}")
print(f"   Must notify by  : {sla['deadline']}")
print(f"   Hours remaining : {sla['hours_remaining']}")
print(f"   Status          : {sla['status']}")

# Boundary checks — an SLA clock that is wrong at the boundary is worse than
# no clock, because it is trusted.
base = datetime(2026, 1, 1, 12, 0, 0)
assert check_breach_sla(base, now=base)["hours_remaining"] == 72.0
assert check_breach_sla(base, now=base + timedelta(hours=71))["status"] == "URGENT"
assert check_breach_sla(base, now=base + timedelta(hours=59))["status"] == "ON TRACK"
assert check_breach_sla(base, now=base + timedelta(hours=73))["status"] == "OVERDUE"
# Exactly at 72h the window has closed, not "just barely open".
assert check_breach_sla(base, now=base + timedelta(hours=72))["status"] == "OVERDUE", \
    "at exactly 72 hours the deadline has passed — off-by-one here means you " \
    "report late while your dashboard says you are fine"
print("\n✅ SLA clock boundary checks pass (0h, 59h, 71h, 72h, 73h)")

print("\n💡 In production, page the DPO the instant a breach is *suspected*, and")
print("   note three things this toy clock does not model:")
print("   • 'Aware' has a legal meaning — a reasonable degree of certainty that a")
print("     breach occurred. A vague alert does not start the clock; a confirmed")
print("     incident does, and you cannot stall the clock by declining to look.")
print("   • Not every breach is notifiable. Art. 33 exempts breaches unlikely to")
print("     result in a risk — but you must document that assessment either way.")
print("   • Art. 34 is a SEPARATE obligation to the data subjects, triggered by")
print("     HIGH risk, with no 72-hour figure at all: 'without undue delay'.")


## 8. Why Microsoft Builds Compliance Tools Into Azure

### Azure's Compliance Portfolio

| Azure Service | What It Does | Mapped to Our Lab |
|--------------|-------------|-------------------|
| **Microsoft Purview Data Map** | Discovers and classifies data across services | Our `generate_residency_map()` |
| **Purview Compliance Manager** | Produces a *readiness score* against control frameworks | Our `generate_controls_report()` — and note we deliberately removed the score |
| **Azure Policy** (`allowedLocations`) | **Prevents** resources being created outside approved regions | Our `detect_violations()` — except Policy *prevents*, we only *detect* |
| **Azure Monitor / resource logs** | Tracks control-plane and data-plane activity | Our `data_residency_log` table |
| **Azure Key Vault / Managed HSM** | Manages encryption keys, optionally per region | Not covered |

The Azure Policy row is the important one. **Detection is a consolation prize.**
Our `detect_violations()` tells you a Swedish user's data is in the wrong
database *after* it is already there — which, for a genuine cross-border
transfer, means the infringement has already occurred and may already be
reportable. A deny-by-default policy at the control plane stops it happening. If
you take one operational idea from this notebook, make it that one: **prefer the
control that prevents over the report that notices.**

### The Business Value

Enterprise customers pay premium prices for Azure because:
1. **Prebuilt tooling** saves months of engineering effort on discovery, classification and evidence collection.
2. **Shared responsibility** — Microsoft is responsible for the compliance of the infrastructure it operates; you remain the **controller** for the personal data you put on it, and the controller carries the accountability obligation (Article 5(2)). Buying compliant infrastructure does not make you compliant.
3. **Certifications and attestations** — Azure holds ISO 27001, ISO 27701, SOC 1/2/3 and many others, and Microsoft offers GDPR-specific **contractual commitments** in its Data Protection Addendum, plus the EU Data Boundary. But **"Azure is pre-certified for GDPR" is not a real thing.** GDPR certification under Article 42 requires a scheme approved by a supervisory authority or the EDPB and an accredited certification body; even where such certification exists, Article 42(4) says it *"does not reduce the responsibility of the controller"*. No cloud provider can be certified compliant on your behalf, because compliance is a property of *your* processing, not of the machines.
4. **Audit support** — Azure produces evidence and reports. You still have to interpret them and connect them to your own processing activities.

The tools in this notebook are toy versions of the *detection* half. Note that the genuinely valuable half is **prevention** (Azure Policy denying non-approved regions, encryption by default, access control), and that the hardest half is neither — it is the documentation and governance no tool can produce for you.

## 🎯 Key Takeaways

1. **Audit where the data IS, not what the data says about itself.** The label-based detector passed a Swedish record sitting in the wrong database because the record's own `home_region` column was honest about where it *should* be. Self-description is not evidence.
2. **Never trust a status field written by the process it is meant to verify.** `erasure_requests.status = 'completed'` was true and the data was still there. Cross-check claims against the data, in every region.
3. **Key every cross-region audit on a stable identifier.** Local `SERIAL` ids differ per database; joining on them merges strangers and splits individuals.
4. **Flag divergence rather than resolving it silently.** When two regions disagree about a field, an audit that keeps whichever value it read first will confidently report a world that does not exist.
5. **Consent is one lawful basis of six.** "Consent rate" is not a compliance metric, and a user without consent is not automatically a problem. Audit that each *purpose* has a declared basis.
6. **Controls are conjunctive; never average them into a score.** One unlawful transfer is an infringement no matter what the other three checks say. A grade implies an authority a script does not have.
7. **Always state scope next to a green tick.** The most dangerous output any compliance tool can produce is a pass with unstated boundaries.
8. **Prefer prevention over detection.** Azure Policy refusing to create a resource outside approved regions beats a report telling you the transfer already happened.
9. **The 72-hour clock starts at awareness** (Art. 33), which means poor detection does not buy you time — it starts your clock late. Art. 34 notification to data subjects is a separate obligation with no 72-hour figure.
10. **None of this makes anything compliant.** These are technical controls that *support* a compliance position built from lawful bases, records, contracts, assessments and governance.

## 🏁 Lab Complete!

You've now built a complete GDPR compliance system using the Azure Paired Regions pattern:

| Notebook | What You Built | And what it showed you was broken |
|----------|---------------|-----------------------------------|
| 1. Data Residency | Geo-routing writes to the correct jurisdiction | A residency claim that only queried its own two rows |
| 2. Cross-Region Replication | Async replication with probe-driven failover | A write acknowledged to the user and then lost; a `health_check` that was never called |
| 3. Right to Erasure | Article 17 across regions, backups and suppression lists | An erasure marked "completed" while the replica still held everything; "anonymisation" that was pseudonymisation |
| 4. Data Sovereignty Audit | Technical controls report with explicit scope | An audit that read a self-reported label and gave a Grade A to a system with an unfulfilled erasure in it |

The pattern across all four: **the version that ran without errors was the
version that was wrong.** Every one of those defects executed cleanly and printed
a reassuring ✅. If there is one habit to take away, it is to make the lab fail
loudly when it stops demonstrating its own lesson — which is why nearly every
section above now ends in an assertion.

### Clean Up

```bash
cd 08-enterprise/gdpr-paired-regions
docker compose down -v  # Stops containers and removes data volumes
```